# PS6E4 Irrigation Prediction: 14-Model GBDT Ensemble

## Method

My approach follows the [KGMON Playbook](https://www.kaggle.com/competitions/playground-series-s6e3/writeups/1st-place-gpt5-4-gemini3-1-claudeopus4-6-kgm)
by Chris Deotte (1st place PS6E3). The core idea: systematic feature engineering,
diverse GBDT models, and multi-level ensembling.

1. Feature engineering with magic formula (see [here](https://www.kaggle.com/competitions/playground-series-s6e4/discussion/687460)), domain interactions, 2-way target encoding
2. 14 GBDT models (XGBoost, LightGBM, CatBoost) with seed and feature diversity
3. Hill climbing ensemble with differential evolution threshold optimization

All training was done on AWS SageMaker spot instances configured via the AWS CLI.
Code was generated with Claude Code (Opus 4.6).

Sections 1-2 (EDA, Features) run live. Section 3 (Training) shows code only.
Section 4 (Ensemble) runs live using pre-computed OOF predictions uploaded on a Kaggle dataset.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import combinations

from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from scipy.optimize import differential_evolution


## 1. EDA

The dataset has 630K synthetic training rows generated from a 10K original dataset.
Target is heavily imbalanced: Low 58.7%, Medium 37.9%, High 3.3%.
Since the metric is balanced accuracy, getting the rare High class right matters
as much as the majority Low class.


In [ ]:
# Paths: adjust for Kaggle vs local
import os
KAGGLE_INPUT = Path("/kaggle/input")
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT / "competitions" / "playground-series-s6e4"
    PRED_DIR = KAGGLE_INPUT / "datasets" / "wguesdon" / "ps6e4-irrigation-14-model-predictions"
    ORIG_PATH = PRED_DIR / "irrigation_prediction.csv"
else:
    DATA_DIR = Path("data/raw")
    PRED_DIR = Path("predictions")
    ORIG_PATH = DATA_DIR / "irrigation_prediction.csv"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
orig = pd.read_csv(ORIG_PATH)

TARGET = "Irrigation_Need"
NUMS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm",
]
CATS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
]

print(f"Train: {train.shape}, Test: {test.shape}, Original: {orig.shape}")
print(f"\nTarget distribution:")
print(train[TARGET].value_counts(normalize=True).round(3))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Target distribution
train[TARGET].value_counts().plot.bar(
    ax=axes[0], color=["#4C72B0", "#DD8452", "#C44E52"]
)
axes[0].set_title("Target Distribution (Train)")
axes[0].set_ylabel("Count")

# Original vs Synthetic distribution
orig_dist = orig[TARGET].value_counts(normalize=True).sort_index()
train_dist = train[TARGET].value_counts(normalize=True).sort_index()
pd.DataFrame({"Original (10K)": orig_dist, "Synthetic (630K)": train_dist}).plot.bar(
    ax=axes[1]
)
axes[1].set_title("Original vs Synthetic")
axes[1].set_ylabel("Proportion")

# Numeric feature std (proxy for information content)
train[NUMS].std().sort_values().plot.barh(ax=axes[2], color="#4C72B0")
axes[2].set_title("Numeric Feature Std Dev")

plt.tight_layout()
plt.show()


In [ ]:
# Top numeric features by class separation
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
top_nums = ["Soil_Moisture", "Wind_Speed_kmh", "Temperature_C",
            "Rainfall_mm", "Humidity", "Sunlight_Hours"]

for ax, col in zip(axes.flat, top_nums):
    for label in ["Low", "Medium", "High"]:
        subset = train[train[TARGET] == label][col]
        ax.hist(subset, bins=50, alpha=0.5, label=label, density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle("Top Numeric Features by Irrigation Class", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))
corr = train[NUMS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, ax=ax)
ax.set_title("Numeric Feature Correlation")
plt.tight_layout()
plt.show()


### Key Discovery: The Magic Formula

The original 10K dataset was generated by a deterministic rule that perfectly
separates all 3 classes (balanced accuracy = 1.0). This was shared by
Chris Deotte in the [forum](https://www.kaggle.com/competitions/playground-series-s6e4/discussion/687460).

The synthetic generator "blurred" these boundaries by adding continuous noise
and label flips. The models learn to correct for this blur.

In [ ]:
# Magic formula: achieves BA = 1.0 on the original 10K dataset
def compute_magic_score(df):
    """Compute the magic formula score from Chris Deotte's discovery."""
    high_score = (
        (df["Soil_Moisture"] < 25).astype(int) * 2
        + (df["Rainfall_mm"] < 300).astype(int) * 2
        + (df["Temperature_C"] > 30).astype(int)
        + (df["Wind_Speed_kmh"] > 10).astype(int)
    )
    low_score = (
        (df["Crop_Growth_Stage"] == "Harvest").astype(int) * 2
        + (df["Crop_Growth_Stage"] == "Sowing").astype(int) * 2
        + (df["Mulching_Used"] == "Yes").astype(int)
    )
    return high_score - low_score


orig["magic_score"] = compute_magic_score(orig)
train["magic_score"] = compute_magic_score(train)

# Verify perfect accuracy on original data
le = LabelEncoder()
le.fit(["High", "Low", "Medium"])
orig_labels = le.transform(orig[TARGET])
orig_preds = np.where(orig["magic_score"] <= 0, 1,
             np.where(orig["magic_score"] >= 4, 0, 2))
orig_ba = balanced_accuracy_score(orig_labels, orig_preds)
print(f"Magic formula on original data: BA = {orig_ba:.4f}")

# On synthetic data
train_labels = le.transform(train[TARGET])
train_preds = np.where(train["magic_score"] <= 0, 1,
              np.where(train["magic_score"] >= 4, 0, 2))
train_ba = balanced_accuracy_score(train_labels, train_preds)
print(f"Magic formula on synthetic data: BA = {train_ba:.4f}")

# Distribution of magic scores by class
fig, ax = plt.subplots(figsize=(10, 4))
for label in ["Low", "Medium", "High"]:
    subset = train[train[TARGET] == label]["magic_score"]
    ax.hist(subset, bins=range(-6, 8), alpha=0.5, label=label, density=True)
ax.set_title("Magic Score Distribution by Class")
ax.set_xlabel("Magic Score")
ax.legend()
plt.tight_layout()
plt.show()


## 2. Feature Engineering (Code Reference)

Our full feature pipeline generates ~300 features from the 19 base columns.
The code below shows the complete pipeline. On Kaggle this would run for
several minutes on the full dataset.

**Feature groups:**
- Magic formula features (15): threshold flags, composite scores, boundary distances
- Domain interactions (13): water balance, heat stress, evapotranspiration
- Digit extraction (11): first decimal digit per numeric column
- Frequency encoding (8): category frequency as numeric
- Boolean flags (3): rainfed, low moisture, high temperature
- feature-engine transformers (41): discretisers, count-frequency encoder
- Original dataset TE priors (19): target mean from 10K original per feature
- 2-way interaction target encoding (118): all C(19,2) pairs, encoded inside CV


In [ ]:
def create_features(df):
    """Full feature engineering pipeline.

    This function creates all non-CV features. The 2-way interaction target
    encoding is applied inside each CV fold to prevent leakage.
    """
    out = df.copy()

    # --- Magic formula features ---
    out["magic_soil_dry"] = (out["Soil_Moisture"] < 25).astype(np.int8)
    out["magic_rain_low"] = (out["Rainfall_mm"] < 300).astype(np.int8)
    out["magic_temp_hot"] = (out["Temperature_C"] > 30).astype(np.int8)
    out["magic_wind_high"] = (out["Wind_Speed_kmh"] > 10).astype(np.int8)
    out["magic_harvest"] = (out["Crop_Growth_Stage"] == "Harvest").astype(np.int8)
    out["magic_sowing"] = (out["Crop_Growth_Stage"] == "Sowing").astype(np.int8)
    out["magic_mulch_yes"] = (out["Mulching_Used"] == "Yes").astype(np.int8)

    out["magic_high_score"] = (
        out["magic_soil_dry"] * 2 + out["magic_rain_low"] * 2
        + out["magic_temp_hot"] + out["magic_wind_high"]
    )
    out["magic_low_score"] = (
        out["magic_harvest"] * 2 + out["magic_sowing"] * 2 + out["magic_mulch_yes"]
    )
    out["magic_score"] = out["magic_high_score"] - out["magic_low_score"]
    out["magic_dist_to_boundary"] = np.minimum(
        np.abs(out["magic_score"] - 0), np.abs(out["magic_score"] - 3)
    )
    out["margin_soil_25"] = out["Soil_Moisture"] - 25
    out["margin_rain_300"] = out["Rainfall_mm"] - 300
    out["margin_temp_30"] = out["Temperature_C"] - 30
    out["margin_wind_10"] = out["Wind_Speed_kmh"] - 10

    # --- Domain interactions ---
    out["water_balance"] = out["Rainfall_mm"] - out["Previous_Irrigation_mm"]
    out["water_per_hectare"] = (
        (out["Rainfall_mm"] + out["Previous_Irrigation_mm"])
        / (out["Field_Area_hectare"] + 0.01)
    )
    out["total_water"] = out["Rainfall_mm"] + out["Previous_Irrigation_mm"]
    out["heat_stress"] = out["Temperature_C"] * (1 - out["Humidity"] / 100)
    out["evapotranspiration"] = (
        out["Temperature_C"] * out["Sunlight_Hours"] / (out["Humidity"] + 1)
    )
    out["drying_index"] = (
        out["Sunlight_Hours"] * out["Wind_Speed_kmh"] / (out["Humidity"] + 1)
    )
    out["temp_wind"] = out["Temperature_C"] * out["Wind_Speed_kmh"]
    out["moisture_deficit"] = 50 - out["Soil_Moisture"]
    out["moisture_temp_ratio"] = out["Soil_Moisture"] / (out["Temperature_C"] + 1)
    out["irrigation_density"] = (
        out["Previous_Irrigation_mm"] / (out["Field_Area_hectare"] + 0.01)
    )

    # --- Frequency encoding ---
    for col in CATS:
        freq = out[col].value_counts(normalize=True)
        out[f"freq_{col}"] = out[col].map(freq)

    # --- Boolean flags ---
    out["is_rainfed"] = (out["Irrigation_Type"] == "Rainfed").astype(np.int8)
    out["low_moisture"] = (out["Soil_Moisture"] < 25).astype(np.int8)
    out["high_temp"] = (out["Temperature_C"] > 35).astype(np.int8)

    # --- Digit extraction ---
    for col in NUMS:
        vals = out[col].values
        frac = vals - np.floor(vals)
        out[f"{col}_d1"] = np.floor(frac * 10).astype(np.int8)

    return out


print("Feature engineering functions defined.")
print("In the full pipeline, these are applied before training.")
print("2-way interaction target encoding happens inside each CV fold.")


## 3. Training Scripts (Not Executed)

Below are the actual training scripts used on AWS SageMaker. Each script runs
a 2-phase pipeline: Optuna hyperparameter tuning followed by 5-fold CV with
the best parameters.

The scripts use [tabml](https://github.com/wguesdon/tabml), a lightweight
model wrapper library. Install with: `pip install git+https://github.com/wguesdon/tabml.git`

### 14 Models trained:

| Model | Algorithm | Seed | Feature Strategy | CV |
|-------|-----------|------|------------------|----|  
| xgb_v2 | XGBoost | 42 | all (516 features) | 0.97208 |
| xgb_s43 | XGBoost | 43 | all | 0.97185 |
| xgb_s44 | XGBoost | 44 | all | 0.97199 |
| xgb_magic_core | XGBoost | 42 | magic_core (~50) | 0.97256 |
| xgb_no_interact | XGBoost | 42 | no_interactions (~130) | 0.97262 |
| lgb_v2 | LightGBM | 42 | all | 0.97140 |
| lgb_s43 | LightGBM | 43 | all | 0.97131 |
| lgb_s44 | LightGBM | 44 | all | 0.97171 |
| cat_v2 | CatBoost | 42 | all | 0.97786 |
| cat_s43 | CatBoost | 43 | all | 0.97802 |
| cat_s44 | CatBoost | 44 | all | 0.97834 |
| cat_magic42 | CatBoost | 42 | all + magic | 0.97804 |
| cat_magic_core | CatBoost | 42 | magic_core | 0.97801 |
| cat_magic_s45 | CatBoost | 45 | all + magic | 0.97812 |

### 3a. Feature Engineering Module (`features.py`)

Shared by all trainers. Uploaded to S3 and loaded dynamically.

In [ ]:
# This code was run on AWS SageMaker, not in this notebook.
# Shown here for full reproducibility.
if False:
    """Feature engineering for PS6E4 AWS SageMaker training.

    This file is uploaded to S3 and loaded dynamically by the trainers.
    It recreates the feature store from 02_feature_engineering.py and
    02b_features_projections.py in a single function, plus the high-value
    techniques from public notebooks (2-way interactions, original TE priors).
    """

    import os
    from itertools import combinations

    import numpy as np
    import pandas as pd
    from sklearn.preprocessing import LabelEncoder


    NUMERIC_COLS = [
        "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
        "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
        "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm",
    ]

    CAT_COLS = [
        "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
        "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
    ]

    ALL_BASE_COLS = NUMERIC_COLS + CAT_COLS

    BIGRAM_PAIRS = [
        ("Crop_Growth_Stage", "Crop_Type"),
        ("Crop_Growth_Stage", "Irrigation_Type"),
        ("Irrigation_Type", "Season"),
        ("Crop_Type", "Season"),
        ("Mulching_Used", "Irrigation_Type"),
        ("Soil_Type", "Crop_Type"),
        ("Region", "Season"),
        ("Irrigation_Type", "Water_Source"),
    ]


    def load_original_data() -> pd.DataFrame:
        """Load the original irrigation dataset from the SageMaker input channel.

        Returns:
            Original dataset DataFrame, or empty DataFrame if not found.
        """
        input_dir = os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training")
        orig_path = os.path.join(input_dir, "irrigation_prediction.csv")
        if os.path.exists(orig_path):
            return pd.read_csv(orig_path)
        return pd.DataFrame()


    def add_original_te_priors(
        train: pd.DataFrame,
        test: pd.DataFrame,
        orig: pd.DataFrame,
        target_col: str = "Irrigation_Need",
    ) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
        """Add target encoding priors from the original dataset.

        For each feature column, computes the mean target value per group
        from the original (10K row) dataset and merges it as a numeric feature.
        Zero leakage since original data is independent of the synthetic split.

        Args:
            train: Training DataFrame (must contain target_col).
            test: Test DataFrame.
            orig: Original dataset DataFrame (must contain target_col).
            target_col: Name of the target column.

        Returns:
            Tuple of (train, test, list of new column names).
        """
        if orig.empty or target_col not in orig.columns:
            return train, test, []

        # Encode target to numeric for mean computation
        target_map = {"Low": 0, "Medium": 1, "High": 2}
        orig_encoded = orig.copy()
        if not pd.api.types.is_numeric_dtype(orig_encoded[target_col]):
            orig_encoded[target_col] = orig_encoded[target_col].map(target_map)
        orig_encoded[target_col] = pd.to_numeric(orig_encoded[target_col], errors="coerce")
        orig_encoded = orig_encoded.dropna(subset=[target_col])

        te_cols = []
        for col in ALL_BASE_COLS:
            if col not in orig_encoded.columns:
                continue
            te_name = f"TE_ORIG_{col}"
            group_means = orig_encoded.groupby(col)[target_col].mean().astype("float32")
            group_means.name = te_name

            train = train.merge(group_means, on=col, how="left")
            train[te_name] = train[te_name].fillna(0.5)
            test = test.merge(group_means, on=col, how="left")
            test[te_name] = test[te_name].fillna(0.5)
            te_cols.append(te_name)

        return train, test, te_cols


    def create_2way_interactions(
        train: pd.DataFrame,
        test: pd.DataFrame,
    ) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
        """Create all 2-way feature interactions as string columns.

        Generates C(19,2) = 171 pairs from all base features. Pairs where
        the number of unique values exceeds half the dataset size are dropped
        (they act like row IDs and add noise).

        These columns are meant to be target-encoded inside the CV loop
        by the trainer, not by features.py.

        Args:
            train: Training DataFrame.
            test: Test DataFrame.

        Returns:
            Tuple of (train, test, list of interaction column names).
        """
        interaction_cols = []

        for col_a, col_b in combinations(ALL_BASE_COLS, 2):
            if col_a not in train.columns or col_b not in train.columns:
                continue

            name = f"{col_a}-{col_b}"
            train_vals = train[col_a].astype(str) + "_" + train[col_b].astype(str)
            test_vals = test[col_a].astype(str) + "_" + test[col_b].astype(str)

            combined = pd.concat([train_vals, test_vals], ignore_index=True)
            combined_encoded, _ = combined.factorize()
            n_unique = pd.Series(combined_encoded).nunique()

            # Drop high-cardinality pairs (act like IDs)
            if n_unique > len(combined) // 2:
                continue

            train[name] = combined_encoded[: len(train)]
            test[name] = combined_encoded[len(train) :]
            interaction_cols.append(name)

        return train, test, interaction_cols


    def create_features(df: pd.DataFrame) -> pd.DataFrame:
        """Create all features for a train or test dataframe.

        This generates the base engineered features. The 2-way interactions
        and original TE priors are handled separately by the trainer since
        they need access to both train and test (or original data).

        Args:
            df: Raw train or test DataFrame with original columns.

        Returns:
            DataFrame with original + engineered columns.
        """
        out = df.copy()

        # Label-encode categoricals
        for col in CAT_COLS:
            if col in out.columns:
                le = LabelEncoder()
                out[f"{col}_enc"] = le.fit_transform(out[col].astype(str))

        # Domain interactions
        out["water_balance"] = out["Rainfall_mm"] - out["Previous_Irrigation_mm"]
        out["water_per_hectare"] = (
            (out["Rainfall_mm"] + out["Previous_Irrigation_mm"])
            / (out["Field_Area_hectare"] + 0.01)
        )
        out["total_water"] = out["Rainfall_mm"] + out["Previous_Irrigation_mm"]
        out["heat_stress"] = out["Temperature_C"] * (1 - out["Humidity"] / 100)
        out["evapotranspiration"] = (
            out["Temperature_C"] * out["Sunlight_Hours"] / (out["Humidity"] + 1)
        )
        out["wind_chill"] = out["Wind_Speed_kmh"] * (100 - out["Humidity"]) / 100
        out["drying_index"] = (
            out["Sunlight_Hours"] * out["Wind_Speed_kmh"] / (out["Humidity"] + 1)
        )
        out["temp_wind_interaction"] = out["Temperature_C"] * out["Wind_Speed_kmh"]
        out["soil_conductivity_ratio"] = (
            out["Electrical_Conductivity"] / (out["Soil_pH"] + 1)
        )
        out["soil_moisture_deficit"] = 50 - out["Soil_Moisture"]
        out["organic_efficiency"] = (
            out["Organic_Carbon"] / (out["Electrical_Conductivity"] + 0.01)
        )
        out["irrigation_density"] = (
            out["Previous_Irrigation_mm"] / (out["Field_Area_hectare"] + 0.01)
        )
        out["rainfall_coverage"] = (
            out["Rainfall_mm"] / (out["Field_Area_hectare"] + 0.01)
        )
        out["moisture_temp_ratio"] = out["Soil_Moisture"] / (out["Temperature_C"] + 1)
        out["moisture_wind_ratio"] = out["Soil_Moisture"] / (out["Wind_Speed_kmh"] + 1)
        out["moisture_rainfall_ratio"] = out["Soil_Moisture"] / (out["Rainfall_mm"] + 1)

        # Frequency encoding for categoricals
        for col in CAT_COLS:
            if col in out.columns:
                freq = out[col].value_counts(normalize=True)
                out[f"freq_{col}"] = out[col].map(freq)

        # Bigram cross-features (label-encoded)
        for col_a, col_b in BIGRAM_PAIRS:
            if col_a in out.columns and col_b in out.columns:
                cross = out[col_a].astype(str) + "__" + out[col_b].astype(str)
                le = LabelEncoder()
                out[f"bi_{col_a}__{col_b}"] = le.fit_transform(cross)
                freq = cross.value_counts(normalize=True)
                out[f"freq_bi_{col_a}__{col_b}"] = cross.map(freq)

        # Boolean flags
        out["is_rainfed"] = (out["Irrigation_Type"] == "Rainfed").astype(np.int8)
        out["is_flowering"] = (out["Crop_Growth_Stage"] == "Flowering").astype(np.int8)
        out["is_harvest"] = (out["Crop_Growth_Stage"] == "Harvest").astype(np.int8)
        out["has_mulching"] = (out["Mulching_Used"] == "Yes").astype(np.int8)
        out["is_loamy"] = (out["Soil_Type"] == "Loamy").astype(np.int8)
        out["is_sandy"] = (out["Soil_Type"] == "Sandy").astype(np.int8)
        out["low_moisture"] = (out["Soil_Moisture"] < 25).astype(np.int8)
        out["high_temp"] = (out["Temperature_C"] > 35).astype(np.int8)
        out["high_wind"] = (out["Wind_Speed_kmh"] > 15).astype(np.int8)
        out["low_rainfall"] = (out["Rainfall_mm"] < 500).astype(np.int8)
        out["stress_count"] = (
            out["low_moisture"] + out["high_temp"] + out["high_wind"] + out["low_rainfall"]
        )

        # ---------------------------------------------------------------
        # Magic Score: exact formula from the original dataset generator
        # Achieves BA=1.0 on original data. On synthetic data the generator
        # "blurred" the boundaries, so we expose the score and its components
        # as features for the model to learn the noise pattern.
        # ---------------------------------------------------------------
        # Individual boolean flags from the formula (exact thresholds)
        out["magic_soil_dry"] = (out["Soil_Moisture"] < 25).astype(np.int8)
        out["magic_rain_low"] = (out["Rainfall_mm"] < 300).astype(np.int8)
        out["magic_temp_hot"] = (out["Temperature_C"] > 30).astype(np.int8)
        out["magic_wind_high"] = (out["Wind_Speed_kmh"] > 10).astype(np.int8)
        out["magic_harvest"] = (out["Crop_Growth_Stage"] == "Harvest").astype(np.int8)
        out["magic_sowing"] = (out["Crop_Growth_Stage"] == "Sowing").astype(np.int8)
        out["magic_mulch_yes"] = (out["Mulching_Used"] == "Yes").astype(np.int8)

        # Composite scores
        out["magic_high_score"] = (
            out["magic_soil_dry"] * 2
            + out["magic_rain_low"] * 2
            + out["magic_temp_hot"]
            + out["magic_wind_high"]
        )
        out["magic_low_score"] = (
            out["magic_harvest"] * 2
            + out["magic_sowing"] * 2
            + out["magic_mulch_yes"]
        )
        out["magic_score"] = out["magic_high_score"] - out["magic_low_score"]

        # Distance to decision boundaries (score 0 and 3 are the boundaries)
        out["magic_dist_to_boundary"] = np.minimum(
            np.abs(out["magic_score"] - 0.5),
            np.abs(out["magic_score"] - 3.5),
        )
        # Continuous margin: how far the raw numeric is from the threshold
        out["margin_soil_25"] = out["Soil_Moisture"] - 25.0
        out["margin_rain_300"] = out["Rainfall_mm"] - 300.0
        out["margin_temp_30"] = out["Temperature_C"] - 30.0
        out["margin_wind_10"] = out["Wind_Speed_kmh"] - 10.0

        # Digit extraction on key numeric columns
        for col in NUMERIC_COLS:
            if col in out.columns:
                vals = out[col].values
                frac = vals - np.floor(vals)
                out[f"{col}_d1"] = np.floor(frac * 10).astype(np.int8)
                out[f"{col}_frac100"] = np.round(frac * 100).astype(np.int16)

        return out


    def get_feature_columns() -> list[str]:
        """Return the list of feature columns after create_features."""
        exclude = {"id", "Irrigation_Need"} | set(CAT_COLS)
        sample = pd.DataFrame({col: [0.0] for col in NUMERIC_COLS})
        for col in CAT_COLS:
            sample[col] = ["dummy"]
        sample["id"] = [0]
        sample["Irrigation_Need"] = ["Low"]
        result = create_features(sample)
        return [c for c in result.columns if c not in exclude]


    def get_feature_columns_from_df(df: pd.DataFrame) -> list[str]:
        """Get feature columns from a processed DataFrame."""
        exclude = {"id", "Irrigation_Need"} | set(CAT_COLS)
        return [c for c in df.columns if c not in exclude]


    # ---------------------------------------------------------------
    # Feature subset strategies for model diversity
    # ---------------------------------------------------------------

    # Named feature groups for subsetting
    FEATURE_GROUPS = {
        "magic": [
            "magic_soil_dry", "magic_rain_low", "magic_temp_hot", "magic_wind_high",
            "magic_harvest", "magic_sowing", "magic_mulch_yes",
            "magic_high_score", "magic_low_score", "magic_score",
            "magic_dist_to_boundary",
            "margin_soil_25", "margin_rain_300", "margin_temp_30", "margin_wind_10",
        ],
        "domain": [
            "water_balance", "water_per_hectare", "total_water", "heat_stress",
            "evapotranspiration", "wind_chill", "drying_index", "temp_wind_interaction",
            "soil_conductivity_ratio", "soil_moisture_deficit", "organic_efficiency",
            "irrigation_density", "rainfall_coverage", "moisture_temp_ratio",
            "moisture_wind_ratio", "moisture_rainfall_ratio",
        ],
        "digits": [f"{c}_d1" for c in NUMERIC_COLS] + [f"{c}_frac100" for c in NUMERIC_COLS],
        "encoded": [f"{c}_enc" for c in CAT_COLS],
        "freq": [f"freq_{c}" for c in CAT_COLS],
        "boolean": [
            "is_rainfed", "is_flowering", "is_harvest", "has_mulching",
            "is_loamy", "is_sandy", "low_moisture", "high_temp",
            "high_wind", "low_rainfall", "stress_count",
        ],
    }


    def get_feature_strategy(strategy: str, all_features: list[str]) -> dict:
        """Return feature selection config for a named diversity strategy.

        Each strategy returns a dict with keys that map to model-native parameters:
          - "keep": list of features to keep (None = keep all)
          - "drop": list of features to drop (applied after keep)
          - "weights": dict mapping feature name to sampling weight (XGBoost)
          - "description": human-readable description

        Args:
            strategy: Strategy name. Options:
                "all" - Use all features (baseline)
                "magic_core" - Magic score + margins + encoded cats only (~40 features)
                "no_magic" - All features except magic score group (diversity)
                "no_interactions" - Drop 2-way interaction columns
                "no_digits" - Drop digit extraction features
                "domain_heavy" - Boost domain + magic features, suppress digits
                "magic_heavy" - Boost magic features, suppress domain
            all_features: Full list of available feature column names.

        Returns:
            Dict with strategy configuration.
        """
        strategies = {
            "all": {
                "keep": None,
                "drop": [],
                "weights": None,
                "description": "All features (baseline)",
            },
            "magic_core": {
                "keep": (
                    FEATURE_GROUPS["magic"]
                    + FEATURE_GROUPS["encoded"]
                    + FEATURE_GROUPS["freq"]
                    + NUMERIC_COLS
                ),
                "drop": [],
                "weights": None,
                "description": "Magic score + raw numerics + encoded cats (~50 features)",
            },
            "no_magic": {
                "keep": None,
                "drop": FEATURE_GROUPS["magic"],
                "weights": None,
                "description": "All features except magic score group",
            },
            "no_interactions": {
                "keep": None,
                "drop": [f for f in all_features if "-" in f],
                "weights": None,
                "description": "Drop 2-way interaction columns",
            },
            "no_digits": {
                "keep": None,
                "drop": FEATURE_GROUPS["digits"],
                "weights": None,
                "description": "Drop digit extraction features",
            },
            "domain_heavy": {
                "keep": None,
                "drop": [],
                "weights": {
                    f: (3.0 if f in FEATURE_GROUPS["magic"] + FEATURE_GROUPS["domain"]
                        else 0.3 if f in FEATURE_GROUPS["digits"]
                        else 1.0)
                    for f in all_features
                },
                "description": "3x weight on magic+domain, 0.3x on digits",
            },
            "magic_heavy": {
                "keep": None,
                "drop": [],
                "weights": {
                    f: (5.0 if f in FEATURE_GROUPS["magic"]
                        else 0.5 if f in FEATURE_GROUPS["domain"]
                        else 1.0)
                    for f in all_features
                },
                "description": "5x weight on magic features, 0.5x on domain",
            },
        }

        if strategy not in strategies:
            raise ValueError(
                f"Unknown strategy '{strategy}'. "
                f"Available: {list(strategies.keys())}"
            )

        config = strategies[strategy]

        # Resolve keep/drop to final feature list
        if config["keep"] is not None:
            features = [f for f in config["keep"] if f in all_features]
        else:
            features = list(all_features)
        features = [f for f in features if f not in config["drop"]]

        config["features"] = features
        return config


    def apply_feature_strategy_to_model(strategy_config: dict, model_type: str) -> dict:
        """Convert a feature strategy config into model-specific parameters.

        Args:
            strategy_config: Output from get_feature_strategy().
            model_type: One of "xgb", "lgb", "cat".

        Returns:
            Dict of model parameters to merge into the model config.
        """
        params = {}
        weights = strategy_config.get("weights")
        features = strategy_config.get("features", [])

        if model_type == "xgb" and weights:
            # XGBoost: feature_weights controls per-feature sampling probability
            import numpy as _np
            weight_array = _np.array([weights.get(f, 1.0) for f in features])
            params["feature_weights"] = weight_array

        elif model_type == "lgb" and weights:
            # LightGBM: use feature_fraction_bynode for global diversity,
            # no native per-feature weights. Use a lower fraction instead.
            params["feature_fraction_bynode"] = 0.5

        elif model_type == "cat" and weights:
            # CatBoost: no native per-feature weights.
            # Use rsm (random subspace method) for column sampling diversity.
            params["rsm"] = 0.5

        return params


### 3b. XGBoost Trainer (`train_xgb_v3.py`)

GPU-accelerated via `xgb.DMatrix`. Uses `compute_sample_weight("balanced")` for class imbalance.

In [ ]:
# This code was run on AWS SageMaker, not in this notebook.
# Shown here for full reproducibility.
if False:
    #!/usr/bin/env python
    """XGBoost SageMaker training entry point for PS6E4 (multiclass).

    Uses tabml.models.XGBoostModel for training. Includes 2-way interaction
    target encoding inside the CV loop and original dataset TE priors.
    """

    import os
    import json
    import numpy as np
    import pandas as pd
    import joblib
    import optuna
    from datetime import datetime
    from pathlib import Path

    from sklearn.model_selection import StratifiedKFold, train_test_split
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.preprocessing import LabelEncoder, TargetEncoder
    from sklearn.utils.class_weight import compute_sample_weight

    from tabml.models import XGBoostModel


    def load_config():
        """Load competition config from SageMaker input channel."""
        input_dir = os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training")
        config_path = os.path.join(input_dir, "config.yaml")
        import yaml
        with open(config_path) as f:
            return yaml.safe_load(f)


    def load_data():
        """Load data with original TE priors and 2-way interactions."""
        import importlib.util
        input_dir = os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training")

        train = pd.read_csv(os.path.join(input_dir, "train.csv"))
        test = pd.read_csv(os.path.join(input_dir, "test.csv"))

        spec = importlib.util.spec_from_file_location(
            "features", os.path.join(input_dir, "features.py")
        )
        features_module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(features_module)

        # Load original data for TE priors
        orig = features_module.load_original_data()

        # Base feature engineering
        train = features_module.create_features(train)
        test = features_module.create_features(test)

        # Add original dataset TE priors (no leakage)
        train, test, te_prior_cols = features_module.add_original_te_priors(
            train, test, orig
        )
        print(f"  Original TE priors: {len(te_prior_cols)} columns")

        # Add 2-way interaction columns (will be target-encoded in CV loop)
        train, test, interaction_cols = features_module.create_2way_interactions(
            train, test
        )
        print(f"  2-way interactions: {len(interaction_cols)} columns")

        feature_cols = features_module.get_feature_columns_from_df(train)

        target_col = "Irrigation_Need"
        le = LabelEncoder()
        le.fit(["Low", "Medium", "High"])
        y = le.transform(train[target_col])

        # Load original data for appending to training folds
        orig_X = None
        orig_y = None
        if not orig.empty and target_col in orig.columns:
            orig = features_module.create_features(orig)
            # Add TE priors to orig too (self-referential but consistent)
            orig_feature_cols = [c for c in feature_cols if c in orig.columns]
            missing_cols = set(feature_cols) - set(orig.columns)
            for col in missing_cols:
                orig[col] = 0
            orig_X = orig[feature_cols]
            orig_y = le.transform(orig[target_col])
            print(f"  Original data: {orig_X.shape[0]} rows appended to training folds")

        return (
            train[feature_cols], test[feature_cols], y, feature_cols,
            test["id"], interaction_cols, orig_X, orig_y,
        )


    def detect_gpu() -> bool:
        """Check for GPU hardware."""
        try:
            import subprocess
            result = subprocess.run(
                ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                capture_output=True, text=True, timeout=5,
            )
            if result.returncode == 0 and result.stdout.strip():
                print(f"GPU detected: {result.stdout.strip()}")
                return True
        except Exception:
            pass
        return False


    def apply_target_encoding(X_tr, X_va, X_te, y_tr, te_cols, seed=42):
        """Apply TargetEncoder to interaction columns inside the fold.

        Args:
            X_tr: Training features for this fold.
            X_va: Validation features for this fold.
            X_te: Test features.
            y_tr: Training target for this fold.
            te_cols: List of interaction column names to encode.
            seed: Random state.

        Returns:
            Tuple of (X_tr, X_va, X_te) with TE columns replaced by encoded values.
        """
        if not te_cols:
            return X_tr, X_va, X_te

        te_cols_present = [c for c in te_cols if c in X_tr.columns]
        if not te_cols_present:
            return X_tr, X_va, X_te

        encoder = TargetEncoder(target_type="multiclass", cv=5, random_state=seed)

        te_train = pd.DataFrame(
            encoder.fit_transform(X_tr[te_cols_present], y_tr),
            index=X_tr.index,
        )
        te_train.columns = encoder.get_feature_names_out(te_cols_present)

        te_val = pd.DataFrame(
            encoder.transform(X_va[te_cols_present]),
            index=X_va.index,
        )
        te_val.columns = encoder.get_feature_names_out(te_cols_present)

        te_test = pd.DataFrame(
            encoder.transform(X_te[te_cols_present]),
            index=X_te.index,
        )
        te_test.columns = encoder.get_feature_names_out(te_cols_present)

        # Replace raw interaction cols with encoded versions
        X_tr = pd.concat(
            [X_tr.drop(columns=te_cols_present).reset_index(drop=True),
             te_train.reset_index(drop=True)], axis=1,
        )
        X_va = pd.concat(
            [X_va.drop(columns=te_cols_present).reset_index(drop=True),
             te_val.reset_index(drop=True)], axis=1,
        )
        X_te = pd.concat(
            [X_te.drop(columns=te_cols_present).reset_index(drop=True),
             te_test.reset_index(drop=True)], axis=1,
        )

        return X_tr, X_va, X_te


    def run():
        """Main training pipeline."""
        model_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
        config = load_config()
        n_folds = 5
        n_trials = config.get("models", {}).get("xgboost", {}).get("n_trials", 50)
        seed = 42
        use_gpu = detect_gpu()

        # SageMaker hyperparameter overrides
        sm_seed = os.environ.get("SM_HP_RANDOM_STATE")
        if sm_seed:
            seed = int(sm_seed)
        sm_trials = os.environ.get("SM_HP_N_TRIALS")
        if sm_trials:
            n_trials = int(sm_trials)
        feature_strategy = os.environ.get("SM_HP_FEATURE_STRATEGY", "all")

        print("=" * 60)
        print("  XGBoost Training v3 (PS6E4 multiclass)")
        print("=" * 60)

        X, X_test, y, feature_cols, test_ids, interaction_cols, orig_X, orig_y = load_data()

        # Apply feature strategy for diversity
        strategy_params = {}
        if feature_strategy != "all":
            import importlib.util
            input_dir = os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training")
            spec = importlib.util.spec_from_file_location(
                "features", os.path.join(input_dir, "features.py")
            )
            fm = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(fm)
            strat = fm.get_feature_strategy(feature_strategy, feature_cols)
            kept = strat["features"]
            X = X[[c for c in kept if c in X.columns]]
            X_test = X_test[[c for c in kept if c in X_test.columns]]
            if orig_X is not None:
                orig_X = orig_X[[c for c in kept if c in orig_X.columns]]
            feature_cols = list(X.columns)
            interaction_cols = [c for c in interaction_cols if c in feature_cols]
            strategy_params = fm.apply_feature_strategy_to_model(strat, "xgb")
            print(f"  Feature strategy: {feature_strategy} ({strat['description']})")

        print(f"Train: {X.shape}, Test: {X_test.shape}")

        # Phase 1: Optuna (no TE for speed, raw interactions as integers)
        print(f"\nPHASE 1: Optuna ({n_trials} trials)")
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=seed
        )
        # Apply TE for tuning split
        X_tr_te, X_val_te, _ = apply_target_encoding(
            X_tr, X_val, X_test, pd.Series(y_tr), interaction_cols, seed
        )

        def objective(trial):
            params = {
                "objective": "multi:softprob",
                "num_class": 3,
                "eval_metric": "mlogloss",
                "max_depth": trial.suggest_int("max_depth", 4, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
                "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "gamma": trial.suggest_float("gamma", 0, 5),
                "max_bin": 1024,
                "random_state": seed,
                "n_jobs": -1,
                "tree_method": "hist",
                "device": "cuda" if use_gpu else "cpu",
                "verbosity": 0,
            }
            model = XGBoostModel(params)
            model.fit(X_tr_te, pd.Series(y_tr), X_val_te, pd.Series(y_val))
            proba = model.predict_proba(X_val_te)
            return balanced_accuracy_score(y_val, proba.argmax(axis=1))

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
        best_params = study.best_params
        print(f"Best tuning score: {study.best_value:.5f}")

        # Phase 2: CV with in-fold TE and original data injection
        print(f"\nPHASE 2: {n_folds}-fold CV")
        final_params = {
            **best_params,
            "objective": "multi:softprob",
            "num_class": 3,
            "eval_metric": "mlogloss",
            "n_estimators": 15000,
            "early_stopping_rounds": 500,
            "max_bin": 1024,
            "random_state": seed,
            "n_jobs": -1,
            "tree_method": "hist",
            "device": "cuda" if use_gpu else "cpu",
            "verbosity": 0,
            **{k: v for k, v in strategy_params.items() if k != "feature_weights"},
        }

        kfold = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        oof_preds = np.zeros((len(X), 3))
        test_preds = np.zeros((len(X_test), 3))
        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y)):
            print(f"\n  Fold {fold + 1}/{n_folds}")
            X_tr = X.iloc[train_idx].copy()
            y_tr = y[train_idx].copy()
            X_va = X.iloc[val_idx].copy()
            X_te = X_test.copy()

            # Append original data to training fold
            if orig_X is not None:
                X_tr = pd.concat([X_tr, orig_X], axis=0).reset_index(drop=True)
                y_tr = np.concatenate([y_tr, orig_y])

            # In-fold target encoding
            X_tr, X_va, X_te = apply_target_encoding(
                X_tr, X_va, X_te, pd.Series(y_tr), interaction_cols, seed
            )

            # Balanced sample weights
            weights = compute_sample_weight(class_weight="balanced", y=y_tr)

            model = XGBoostModel(final_params)
            model.fit(X_tr, pd.Series(y_tr), X_va, pd.Series(y[val_idx]))
            oof_preds[val_idx] = model.predict_proba(X_va)
            test_preds += model.predict_proba(X_te) / n_folds

            score = balanced_accuracy_score(y[val_idx], oof_preds[val_idx].argmax(axis=1))
            fold_scores.append(score)
            print(f"  Fold {fold + 1} balanced_accuracy: {score:.5f}")

            joblib.dump(model, os.path.join(model_dir, f"xgb_fold{fold}.pkl"))

        cv_score = balanced_accuracy_score(y, oof_preds.argmax(axis=1))
        print(f"\nCV balanced_accuracy: {cv_score:.5f} (+/- {np.std(fold_scores):.5f})")

        # Save artifacts
        np.save(os.path.join(model_dir, "xgb_oof.npy"), oof_preds)
        np.save(os.path.join(model_dir, "xgb_test.npy"), test_preds)

        results = {
            "model": "xgb",
            "version": "v2",
            "cv_balanced_accuracy": float(cv_score),
            "cv_std": float(np.std(fold_scores)),
            "fold_scores": [float(s) for s in fold_scores],
            "best_params": best_params,
            "n_trials": n_trials,
            "n_folds": n_folds,
            "n_features": X_tr.shape[1],
            "n_interaction_cols": len(interaction_cols),
            "has_orig_te_priors": True,
            "has_orig_data_injection": orig_X is not None,
            "feature_strategy": feature_strategy,
            "seed": seed,
            "timestamp": datetime.now().isoformat(),
        }
        with open(os.path.join(model_dir, "xgb_results.json"), "w") as f:
            json.dump(results, f, indent=2)

        print(f"\nArtifacts saved to {model_dir}")


    if __name__ == "__main__":
        run()


### 3c. LightGBM Trainer (`train_lgb_v3.py`)

CPU-based (leaf-wise growth). Uses `is_unbalance=True` for class imbalance.

In [ ]:
# This code was run on AWS SageMaker, not in this notebook.
# Shown here for full reproducibility.
if False:
    #!/usr/bin/env python
    """LightGBM SageMaker training entry point for PS6E4 (multiclass).

    Uses tabml.models.LightGBMModel. Includes 2-way interaction target encoding
    inside the CV loop and original dataset TE priors.
    """

    import os
    import json
    import numpy as np
    import pandas as pd
    import joblib
    import optuna
    from datetime import datetime

    from sklearn.model_selection import StratifiedKFold, train_test_split
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.preprocessing import LabelEncoder, TargetEncoder

    from tabml.models import LightGBMModel


    def load_config():
        """Load competition config."""
        import yaml
        input_dir = os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training")
        with open(os.path.join(input_dir, "config.yaml")) as f:
            return yaml.safe_load(f)


    def load_data():
        """Load data with original TE priors and 2-way interactions."""
        import importlib.util
        input_dir = os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training")

        train = pd.read_csv(os.path.join(input_dir, "train.csv"))
        test = pd.read_csv(os.path.join(input_dir, "test.csv"))

        spec = importlib.util.spec_from_file_location(
            "features", os.path.join(input_dir, "features.py")
        )
        features_module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(features_module)

        orig = features_module.load_original_data()

        train = features_module.create_features(train)
        test = features_module.create_features(test)

        train, test, te_prior_cols = features_module.add_original_te_priors(
            train, test, orig
        )
        print(f"  Original TE priors: {len(te_prior_cols)} columns")

        train, test, interaction_cols = features_module.create_2way_interactions(
            train, test
        )
        print(f"  2-way interactions: {len(interaction_cols)} columns")

        feature_cols = features_module.get_feature_columns_from_df(train)

        le = LabelEncoder()
        le.fit(["Low", "Medium", "High"])
        y = le.transform(train["Irrigation_Need"])

        orig_X = None
        orig_y = None
        if not orig.empty and "Irrigation_Need" in orig.columns:
            orig = features_module.create_features(orig)
            missing_cols = set(feature_cols) - set(orig.columns)
            for col in missing_cols:
                orig[col] = 0
            orig_X = orig[feature_cols]
            orig_y = le.transform(orig["Irrigation_Need"])
            print(f"  Original data: {orig_X.shape[0]} rows appended to training folds")

        return (
            train[feature_cols], test[feature_cols], y, feature_cols,
            test["id"], interaction_cols, orig_X, orig_y,
        )


    def apply_target_encoding(X_tr, X_va, X_te, y_tr, te_cols, seed=42):
        """Apply TargetEncoder to interaction columns inside the fold."""
        if not te_cols:
            return X_tr, X_va, X_te

        te_cols_present = [c for c in te_cols if c in X_tr.columns]
        if not te_cols_present:
            return X_tr, X_va, X_te

        encoder = TargetEncoder(target_type="multiclass", cv=5, random_state=seed)

        te_train = pd.DataFrame(
            encoder.fit_transform(X_tr[te_cols_present], y_tr),
            index=X_tr.index,
        )
        te_train.columns = encoder.get_feature_names_out(te_cols_present)

        te_val = pd.DataFrame(
            encoder.transform(X_va[te_cols_present]),
            index=X_va.index,
        )
        te_val.columns = encoder.get_feature_names_out(te_cols_present)

        te_test = pd.DataFrame(
            encoder.transform(X_te[te_cols_present]),
            index=X_te.index,
        )
        te_test.columns = encoder.get_feature_names_out(te_cols_present)

        X_tr = pd.concat(
            [X_tr.drop(columns=te_cols_present).reset_index(drop=True),
             te_train.reset_index(drop=True)], axis=1,
        )
        X_va = pd.concat(
            [X_va.drop(columns=te_cols_present).reset_index(drop=True),
             te_val.reset_index(drop=True)], axis=1,
        )
        X_te = pd.concat(
            [X_te.drop(columns=te_cols_present).reset_index(drop=True),
             te_test.reset_index(drop=True)], axis=1,
        )

        return X_tr, X_va, X_te


    def run():
        """Main training pipeline."""
        model_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
        config = load_config()
        n_folds = 5
        n_trials = config.get("models", {}).get("lightgbm", {}).get("n_trials", 50)
        seed = 42

        sm_seed = os.environ.get("SM_HP_RANDOM_STATE")
        if sm_seed:
            seed = int(sm_seed)
        sm_trials = os.environ.get("SM_HP_N_TRIALS")
        if sm_trials:
            n_trials = int(sm_trials)
        feature_strategy = os.environ.get("SM_HP_FEATURE_STRATEGY", "all")

        print("=" * 60)
        print("  LightGBM Training v3 (PS6E4 multiclass)")
        print("=" * 60)

        X, X_test, y, feature_cols, test_ids, interaction_cols, orig_X, orig_y = load_data()

        strategy_params = {}
        if feature_strategy != "all":
            import importlib.util
            input_dir = os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training")
            spec = importlib.util.spec_from_file_location(
                "features", os.path.join(input_dir, "features.py")
            )
            fm = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(fm)
            strat = fm.get_feature_strategy(feature_strategy, feature_cols)
            kept = strat["features"]
            X = X[[c for c in kept if c in X.columns]]
            X_test = X_test[[c for c in kept if c in X_test.columns]]
            if orig_X is not None:
                orig_X = orig_X[[c for c in kept if c in orig_X.columns]]
            feature_cols = list(X.columns)
            interaction_cols = [c for c in interaction_cols if c in feature_cols]
            strategy_params = fm.apply_feature_strategy_to_model(strat, "lgb")
            print(f"  Feature strategy: {feature_strategy} ({strat['description']})")

        print(f"Train: {X.shape}, Test: {X_test.shape}")

        # Phase 1: Optuna
        print(f"\nPHASE 1: Optuna ({n_trials} trials)")
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=seed
        )
        X_tr_te, X_val_te, _ = apply_target_encoding(
            X_tr, X_val, X_test, pd.Series(y_tr), interaction_cols, seed
        )

        def objective(trial):
            params = {
                "objective": "multiclass",
                "num_class": 3,
                "metric": "multi_logloss",
                "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "goss"]),
                "num_leaves": trial.suggest_int("num_leaves", 31, 255),
                "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
                "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "is_unbalance": True,
                "random_state": seed,
                "n_jobs": -1,
                "verbosity": -1,
            }
            model = LightGBMModel(params)
            model.fit(X_tr_te, pd.Series(y_tr), X_val_te, pd.Series(y_val))
            proba = model.predict_proba(X_val_te)
            return balanced_accuracy_score(y_val, proba.argmax(axis=1))

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
        best_params = study.best_params
        print(f"Best tuning score: {study.best_value:.5f}")

        # Phase 2: CV with in-fold TE
        print(f"\nPHASE 2: {n_folds}-fold CV")
        final_params = {
            **best_params,
            "objective": "multiclass",
            "num_class": 3,
            "metric": "multi_logloss",
            "is_unbalance": True,
            "n_estimators": 15000,
            "random_state": seed,
            "n_jobs": -1,
            "verbosity": -1,
            **strategy_params,
        }

        kfold = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        oof_preds = np.zeros((len(X), 3))
        test_preds = np.zeros((len(X_test), 3))
        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y)):
            print(f"\n  Fold {fold + 1}/{n_folds}")
            X_tr = X.iloc[train_idx].copy()
            y_tr = y[train_idx].copy()
            X_va = X.iloc[val_idx].copy()
            X_te = X_test.copy()

            if orig_X is not None:
                X_tr = pd.concat([X_tr, orig_X], axis=0).reset_index(drop=True)
                y_tr = np.concatenate([y_tr, orig_y])

            X_tr, X_va, X_te = apply_target_encoding(
                X_tr, X_va, X_te, pd.Series(y_tr), interaction_cols, seed
            )

            model = LightGBMModel(final_params)
            model.fit(X_tr, pd.Series(y_tr), X_va, pd.Series(y[val_idx]))
            oof_preds[val_idx] = model.predict_proba(X_va)
            test_preds += model.predict_proba(X_te) / n_folds

            score = balanced_accuracy_score(y[val_idx], oof_preds[val_idx].argmax(axis=1))
            fold_scores.append(score)
            print(f"  Fold {fold + 1} balanced_accuracy: {score:.5f}")

            joblib.dump(model, os.path.join(model_dir, f"lgb_fold{fold}.pkl"))

        cv_score = balanced_accuracy_score(y, oof_preds.argmax(axis=1))
        print(f"\nCV balanced_accuracy: {cv_score:.5f} (+/- {np.std(fold_scores):.5f})")

        np.save(os.path.join(model_dir, "lgb_oof.npy"), oof_preds)
        np.save(os.path.join(model_dir, "lgb_test.npy"), test_preds)

        results = {
            "model": "lgb",
            "version": "v2",
            "cv_balanced_accuracy": float(cv_score),
            "cv_std": float(np.std(fold_scores)),
            "fold_scores": [float(s) for s in fold_scores],
            "best_params": best_params,
            "n_trials": n_trials,
            "n_folds": n_folds,
            "n_features": X_tr.shape[1],
            "n_interaction_cols": len(interaction_cols),
            "has_orig_te_priors": True,
            "has_orig_data_injection": orig_X is not None,
            "feature_strategy": feature_strategy,
            "seed": seed,
            "timestamp": datetime.now().isoformat(),
        }
        with open(os.path.join(model_dir, "lgb_results.json"), "w") as f:
            json.dump(results, f, indent=2)

        print(f"\nArtifacts saved to {model_dir}")


    if __name__ == "__main__":
        run()


### 3d. CatBoost Trainer (`train_cat_v3.py`)

GPU-accelerated. Uses `auto_class_weights="Balanced"`. Best single model family (CV 0.978+).

In [ ]:
# This code was run on AWS SageMaker, not in this notebook.
# Shown here for full reproducibility.
if False:
    #!/usr/bin/env python
    """CatBoost SageMaker training entry point for PS6E4 (multiclass).

    Uses tabml.models.CatBoostModel. GPU-accelerated. Includes 2-way interaction
    target encoding inside the CV loop and original dataset TE priors.
    """

    import os
    import json
    import numpy as np
    import pandas as pd
    import optuna
    from datetime import datetime

    from sklearn.model_selection import StratifiedKFold, train_test_split
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.preprocessing import LabelEncoder, TargetEncoder

    from tabml.models import CatBoostModel


    def load_config():
        """Load competition config."""
        import yaml
        input_dir = os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training")
        with open(os.path.join(input_dir, "config.yaml")) as f:
            return yaml.safe_load(f)


    def load_data():
        """Load data with original TE priors and 2-way interactions."""
        import importlib.util
        input_dir = os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training")

        train = pd.read_csv(os.path.join(input_dir, "train.csv"))
        test = pd.read_csv(os.path.join(input_dir, "test.csv"))

        spec = importlib.util.spec_from_file_location(
            "features", os.path.join(input_dir, "features.py")
        )
        features_module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(features_module)

        orig = features_module.load_original_data()

        train = features_module.create_features(train)
        test = features_module.create_features(test)

        train, test, te_prior_cols = features_module.add_original_te_priors(
            train, test, orig
        )
        print(f"  Original TE priors: {len(te_prior_cols)} columns")

        train, test, interaction_cols = features_module.create_2way_interactions(
            train, test
        )
        print(f"  2-way interactions: {len(interaction_cols)} columns")

        feature_cols = features_module.get_feature_columns_from_df(train)

        le = LabelEncoder()
        le.fit(["Low", "Medium", "High"])
        y = le.transform(train["Irrigation_Need"])

        orig_X = None
        orig_y = None
        if not orig.empty and "Irrigation_Need" in orig.columns:
            orig = features_module.create_features(orig)
            missing_cols = set(feature_cols) - set(orig.columns)
            for col in missing_cols:
                orig[col] = 0
            orig_X = orig[feature_cols]
            orig_y = le.transform(orig["Irrigation_Need"])
            print(f"  Original data: {orig_X.shape[0]} rows appended to training folds")

        return (
            train[feature_cols], test[feature_cols], y, feature_cols,
            test["id"], interaction_cols, orig_X, orig_y,
        )


    def detect_gpu() -> bool:
        """Check for GPU hardware."""
        try:
            import subprocess
            result = subprocess.run(
                ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                capture_output=True, text=True, timeout=5,
            )
            if result.returncode == 0 and result.stdout.strip():
                print(f"GPU detected: {result.stdout.strip()}")
                return True
        except Exception:
            pass
        return False


    def apply_target_encoding(X_tr, X_va, X_te, y_tr, te_cols, seed=42):
        """Apply TargetEncoder to interaction columns inside the fold."""
        if not te_cols:
            return X_tr, X_va, X_te

        te_cols_present = [c for c in te_cols if c in X_tr.columns]
        if not te_cols_present:
            return X_tr, X_va, X_te

        encoder = TargetEncoder(target_type="multiclass", cv=5, random_state=seed)

        te_train = pd.DataFrame(
            encoder.fit_transform(X_tr[te_cols_present], y_tr),
            index=X_tr.index,
        )
        te_train.columns = encoder.get_feature_names_out(te_cols_present)

        te_val = pd.DataFrame(
            encoder.transform(X_va[te_cols_present]),
            index=X_va.index,
        )
        te_val.columns = encoder.get_feature_names_out(te_cols_present)

        te_test = pd.DataFrame(
            encoder.transform(X_te[te_cols_present]),
            index=X_te.index,
        )
        te_test.columns = encoder.get_feature_names_out(te_cols_present)

        X_tr = pd.concat(
            [X_tr.drop(columns=te_cols_present).reset_index(drop=True),
             te_train.reset_index(drop=True)], axis=1,
        )
        X_va = pd.concat(
            [X_va.drop(columns=te_cols_present).reset_index(drop=True),
             te_val.reset_index(drop=True)], axis=1,
        )
        X_te = pd.concat(
            [X_te.drop(columns=te_cols_present).reset_index(drop=True),
             te_test.reset_index(drop=True)], axis=1,
        )

        return X_tr, X_va, X_te


    def run():
        """Main training pipeline."""
        model_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
        config = load_config()
        n_folds = 5
        n_trials = config.get("models", {}).get("catboost", {}).get("n_trials", 50)
        seed = 42
        use_gpu = detect_gpu()

        sm_seed = os.environ.get("SM_HP_RANDOM_STATE")
        if sm_seed:
            seed = int(sm_seed)
        sm_trials = os.environ.get("SM_HP_N_TRIALS")
        if sm_trials:
            n_trials = int(sm_trials)
        feature_strategy = os.environ.get("SM_HP_FEATURE_STRATEGY", "all")

        print("=" * 60)
        print("  CatBoost Training v3 (PS6E4 multiclass)")
        print("=" * 60)

        X, X_test, y, feature_cols, test_ids, interaction_cols, orig_X, orig_y = load_data()

        strategy_params = {}
        if feature_strategy != "all":
            import importlib.util
            input_dir = os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training")
            spec = importlib.util.spec_from_file_location(
                "features", os.path.join(input_dir, "features.py")
            )
            fm = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(fm)
            strat = fm.get_feature_strategy(feature_strategy, feature_cols)
            kept = strat["features"]
            X = X[[c for c in kept if c in X.columns]]
            X_test = X_test[[c for c in kept if c in X_test.columns]]
            if orig_X is not None:
                orig_X = orig_X[[c for c in kept if c in orig_X.columns]]
            feature_cols = list(X.columns)
            interaction_cols = [c for c in interaction_cols if c in feature_cols]
            strategy_params = fm.apply_feature_strategy_to_model(strat, "cat")
            print(f"  Feature strategy: {feature_strategy} ({strat['description']})")

        print(f"Train: {X.shape}, Test: {X_test.shape}")

        # Phase 1: Optuna
        print(f"\nPHASE 1: Optuna ({n_trials} trials)")
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=seed
        )
        X_tr_te, X_val_te, _ = apply_target_encoding(
            X_tr, X_val, X_test, pd.Series(y_tr), interaction_cols, seed
        )

        def objective(trial):
            params = {
                "loss_function": "MultiClass",
                "classes_count": 3,
                "auto_class_weights": "Balanced",
                "depth": trial.suggest_int("depth", 4, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
                "iterations": trial.suggest_int("iterations", 300, 2000),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
                "random_strength": trial.suggest_float("random_strength", 0.1, 10.0, log=True),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
                "border_count": trial.suggest_int("border_count", 32, 255),
                "random_seed": seed,
                "verbose": 0,
            }
            if use_gpu:
                params["task_type"] = "GPU"

            model = CatBoostModel(params)
            model.fit(X_tr_te, pd.Series(y_tr), X_val_te, pd.Series(y_val))
            proba = model.predict_proba(X_val_te)
            return balanced_accuracy_score(y_val, proba.argmax(axis=1))

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
        best_params = study.best_params
        print(f"Best tuning score: {study.best_value:.5f}")

        # Phase 2: CV with in-fold TE
        print(f"\nPHASE 2: {n_folds}-fold CV")
        final_params = {
            **best_params,
            "loss_function": "MultiClass",
            "classes_count": 3,
            "auto_class_weights": "Balanced",
            "iterations": 5000,
            "random_seed": seed,
            "verbose": 0,
            **strategy_params,
        }
        if use_gpu:
            final_params["task_type"] = "GPU"

        kfold = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        oof_preds = np.zeros((len(X), 3))
        test_preds = np.zeros((len(X_test), 3))
        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y)):
            print(f"\n  Fold {fold + 1}/{n_folds}")
            X_tr = X.iloc[train_idx].copy()
            y_tr = y[train_idx].copy()
            X_va = X.iloc[val_idx].copy()
            X_te = X_test.copy()

            if orig_X is not None:
                X_tr = pd.concat([X_tr, orig_X], axis=0).reset_index(drop=True)
                y_tr = np.concatenate([y_tr, orig_y])

            X_tr, X_va, X_te = apply_target_encoding(
                X_tr, X_va, X_te, pd.Series(y_tr), interaction_cols, seed
            )

            model = CatBoostModel(final_params)
            model.fit(X_tr, pd.Series(y_tr), X_va, pd.Series(y[val_idx]))
            oof_preds[val_idx] = model.predict_proba(X_va)
            test_preds += model.predict_proba(X_te) / n_folds

            score = balanced_accuracy_score(y[val_idx], oof_preds[val_idx].argmax(axis=1))
            fold_scores.append(score)
            print(f"  Fold {fold + 1} balanced_accuracy: {score:.5f}")

            model.model.save_model(os.path.join(model_dir, f"cat_fold{fold}.cbm"))

        cv_score = balanced_accuracy_score(y, oof_preds.argmax(axis=1))
        print(f"\nCV balanced_accuracy: {cv_score:.5f} (+/- {np.std(fold_scores):.5f})")

        np.save(os.path.join(model_dir, "cat_oof.npy"), oof_preds)
        np.save(os.path.join(model_dir, "cat_test.npy"), test_preds)

        results = {
            "model": "cat",
            "version": "v2",
            "cv_balanced_accuracy": float(cv_score),
            "cv_std": float(np.std(fold_scores)),
            "fold_scores": [float(s) for s in fold_scores],
            "best_params": best_params,
            "n_trials": n_trials,
            "n_folds": n_folds,
            "n_features": X_tr.shape[1],
            "n_interaction_cols": len(interaction_cols),
            "has_orig_te_priors": True,
            "has_orig_data_injection": orig_X is not None,
            "feature_strategy": feature_strategy,
            "seed": seed,
            "timestamp": datetime.now().isoformat(),
        }
        with open(os.path.join(model_dir, "cat_results.json"), "w") as f:
            json.dump(results, f, indent=2)

        print(f"\nArtifacts saved to {model_dir}")


    if __name__ == "__main__":
        run()


## 4. Ensemble: Hill Climbing + Differential Evolution Thresholds

This section runs live. We load the 14 pre-computed OOF and test predictions
from our companion Kaggle dataset, then:
1. Hill climbing to find optimal blend weights
2. Differential evolution to optimize class-specific thresholds for balanced accuracy


In [ ]:
# Load pre-computed predictions
MODEL_NAMES = [
    "xgb_v2", "xgb_s43", "xgb_s44", "xgb_magic_core", "xgb_no_interact",
    "lgb_v2", "lgb_s43", "lgb_s44",
    "cat_v2", "cat_s43", "cat_s44",
    "cat_magic42", "cat_magic_core", "cat_magic_s45",
]

oof_dict = {}
pred_dict = {}
for name in MODEL_NAMES:
    oof_dict[name] = np.load(PRED_DIR / f"oof_{name}.npy")
    pred_dict[name] = np.load(PRED_DIR / f"pred_{name}.npy")

print(f"Loaded {len(MODEL_NAMES)} models")
print(f"OOF shape: {oof_dict[MODEL_NAMES[0]].shape}")
print(f"Test pred shape: {pred_dict[MODEL_NAMES[0]].shape}")

# Load ground truth labels
le = LabelEncoder()
le.fit(["High", "Low", "Medium"])
y = le.transform(train[TARGET])
test_ids = test["id"]

# Individual model CV scores
print("\nIndividual model CV scores:")
for name in MODEL_NAMES:
    score = balanced_accuracy_score(y, oof_dict[name].argmax(axis=1))
    print(f"  {name:20s}  CV = {score:.5f}")


In [ ]:
# Model prediction correlation (diversity check)
corr_data = {}
for name in MODEL_NAMES:
    corr_data[name] = oof_dict[name].argmax(axis=1)
corr_df = pd.DataFrame(corr_data)

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
sns.heatmap(corr_df.corr(), mask=mask, annot=True, fmt=".3f", cmap="coolwarm",
            vmin=0.8, vmax=1.0, ax=ax)
ax.set_title("Model Prediction Correlation (lower = more diverse)")
plt.tight_layout()
plt.show()


### Hill Climbing Ensemble

Greedy weight optimization: perturb weights randomly, keep changes that
improve balanced accuracy on OOF predictions.


In [ ]:
# Hill climbing weight search (10000 iterations)
# Searches for better-than-equal weights across all 14 models.
def hill_climbing_ensemble(oof_dict, y_true, n_iterations=10000, seed=42):
    rng = np.random.RandomState(seed)
    names = list(oof_dict.keys())
    oofs = [oof_dict[n] for n in names]
    n_models = len(oofs)

    # Start with equal weights
    weights = np.ones(n_models) / n_models
    blend = sum(w * o for w, o in zip(weights, oofs))
    best_score = balanced_accuracy_score(y_true, blend.argmax(axis=1))
    best_weights = weights.copy()

    for i in range(n_iterations):
        lr = max(0.003, 0.1 * (1 - i / n_iterations))
        new_weights = best_weights + rng.randn(n_models) * lr
        new_weights = np.clip(new_weights, 0, None)
        new_weights /= new_weights.sum()

        blend = sum(w * o for w, o in zip(new_weights, oofs))
        score = balanced_accuracy_score(y_true, blend.argmax(axis=1))

        if score > best_score:
            best_score = score
            best_weights = new_weights.copy()

    return {n: float(w) for n, w in zip(names, best_weights)}, best_score

weights, ensemble_cv = hill_climbing_ensemble(oof_dict, y, n_iterations=10000)
print(f"Hill climbing ensemble CV: {ensemble_cv:.5f}")
print(f"\nModel weights (non-zero):")
for name, w in sorted(weights.items(), key=lambda x: -x[1]):
    if w > 0.001:
        print(f"  {name:20s}  {w:.4f}")


In [ ]:
# Weight distribution (equal weights)
fig, ax = plt.subplots(figsize=(10, 5))
names_sorted = sorted(MODEL_NAMES)
colors = ["#C44E52" if "cat" in n else "#4C72B0" if "xgb" in n else "#DD8452"
          for n in names_sorted]
ax.barh(names_sorted, [weights[n] for n in names_sorted], color=colors)
ax.set_title("Ensemble Weights (Equal)")
ax.set_xlabel("Weight")
plt.tight_layout()
plt.show()


### Differential Evolution Threshold Optimization

Balanced accuracy = average recall across classes. By applying per-class
multipliers to the blended probabilities before argmax, we can shift the
decision boundaries to maximize this metric.

Differential evolution is a global optimizer that avoids the local optima
that Nelder-Mead can get stuck in.


In [ ]:
from scipy.optimize import differential_evolution

# Create blended OOF and test predictions using optimized weights
oof_blend = sum(weights[n] * oof_dict[n] for n in MODEL_NAMES)
test_blend = sum(weights[n] * pred_dict[n] for n in MODEL_NAMES)

baseline_cv = balanced_accuracy_score(y, oof_blend.argmax(axis=1))
print(f"Ensemble CV (no thresholds): {baseline_cv:.5f}")

# Differential evolution: global optimizer for class thresholds
# Run with high maxiter and tight tolerance for best result.
# Classes: 0=High, 1=Low, 2=Medium (LabelEncoder alphabetical order)
def neg_ba(thresholds):
    return -balanced_accuracy_score(y, (oof_blend * thresholds).argmax(axis=1))

result = differential_evolution(
    neg_ba,
    bounds=[(0.3, 4.0), (0.3, 4.0), (0.3, 4.0)],
    seed=42,
    maxiter=2000,
    popsize=30,
    tol=1e-12,
    mutation=(0.5, 1.5),
    recombination=0.9,
)

best_thresholds = result.x
optimized_cv = -result.fun

print(f"Thresholds: {best_thresholds}")
print(f"Ensemble CV (with thresholds): {optimized_cv:.5f}")
print(f"Gain from threshold optimization: +{optimized_cv - baseline_cv:.5f}")


In [ ]:
# Confusion matrix
optimized_preds = (oof_blend * best_thresholds).argmax(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y, optimized_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["High", "Low", "Medium"],
            yticklabels=["High", "Low", "Medium"])
axes[0].set_title(f"Confusion Matrix (CV={optimized_cv:.5f})")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

# Per-class recall
class_names = ["High", "Low", "Medium"]
recalls = []
for cls in range(3):
    mask = y == cls
    recall = (optimized_preds[mask] == cls).mean()
    recalls.append(recall)
    print(f"  {class_names[cls]:8s} recall = {recall:.4f} (n={mask.sum()})")

axes[1].bar(class_names, recalls, color=["#C44E52", "#4C72B0", "#DD8452"])
axes[1].set_title("Per-Class Recall")
axes[1].set_ylabel("Recall")
axes[1].set_ylim(0.9, 1.0)

plt.tight_layout()
plt.show()


## 5. Submission


In [ ]:
final_test_probs = test_blend * best_thresholds
final_test_classes = final_test_probs.argmax(axis=1)

labels = le.inverse_transform(final_test_classes)
submission = pd.DataFrame({"id": test_ids, TARGET: labels})
submission.to_csv("submission.csv", index=False)

print(f"Submission saved: submission.csv ({len(submission)} rows)")
print(f"\nPredicted distribution:")
print(submission[TARGET].value_counts())


## Summary

| Component | Details |
|-----------|---------|
| Models | 14 GBDT models (5 XGBoost, 3 LightGBM, 6 CatBoost) |
| Diversity | 4 seeds (42-45) + 3 feature strategies (all, magic_core, no_interactions) |
| Features | ~300 features: magic formula, domain interactions, 2-way TE, digit extraction |
| Ensemble | Hill climbing weight optimization (5000 iterations) |
| Thresholds | Differential evolution on per-class probability multipliers |
| Infrastructure | AWS SageMaker spot instances (GPU for XGB/CatBoost, CPU for LightGBM) |
| Best CV | 0.97930 |
| Best LB | 0.97876 |
